# Bulk modulus — volumetric-strain sweep

Analysis of **every** level of a `compress_slab` sweep (drained isotropic compression of an isolated gel at
constant bath pressure: wet NPT-pistons hold the bath, six solvent-transparent plates close together), overlaid so
the whole stress–strain response is visible (`K`, the isotropy of the response, `D_c` vs `ε_vol`).  The counterpart
of `triaxial_compression_sweep_two_pist.ipynb`.  `COMP_LEVELS` must match `STRAIN_TARGETS=(...)` in
`compress_slab.batch`.  To look closely at one level use `bulk_modulus_analysis_single.ipynb`.  Notes at the end
(2026-09-29).

## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: bulk.py (this analysis) on top of triaxial.py (shared machinery)
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
import bulk
tri = importlib.reload(tri)            # pick up edits to lib/*.py without a kernel restart
bulk = importlib.reload(bulk)
bulk.setup_style()
print('analysis code: ', LIB / 'bulk.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
cfg = bulk.Config(
    DATANAME    = "final_config_slab_support_5beads_tall_rho04_new_03_1.0_1.0_10000000_six_plates",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = None,           # <steps> tag = each level's auto-sized HOLD LENGTH; None resolves it from the files
    RUN_ID      = "six_plates_new_03_bulk_sweep_1",  # local folder under flow_data_local/{bulk,plots/bulk}
    COMP_LEVELS = ["0.05", "0.10", "0.15", "0.20", "0.30"],   # = STRAIN_TARGETS=(...) in compress_slab.batch (volumetric strain)
    AXES        = ('z', 'x', 'y'),
    plateau_frac      = 0.25,
    plateau_frac_auto = 0.45,
    P_BARO = 1.5,
    K_SUBTRACT_REF = True, K_STRAIN = 'disp', PLATE_AREA = 'gap',   # K_STRAIN 'disp' = network strain from the cumulative displacement profiles (decks since 2026-10-05; falls back rg -> plate)
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, VOR_EVO_FRAMES = 5, P_CAL = 1.5, P_CAL_MODE = 'const',
    M_REF = None, G_REF = None,
    SS_FIT_NPTS = 3,              # levels in the small-strain slope of figure 7b
    DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_SLOW_REF = 0.17, DC_TARGET_RESID = 0.01,
)
SWELL_ANISO = None            # stress_aniso file of the slab_with_support run (free swelling), for the whole-run anisotropy figure

In [ ]:
SYNC, FORCE_SYNC = True, False
if SYNC:
    bulk.sync_from_expanse(cfg, force=FORCE_SYNC)

In [ ]:
# Load + compute everything
R = bulk.load_reference(cfg)
LEVELS = [L for L in (bulk.load_level(cfg, R, lvl) for lvl in cfg.COMP_LEVELS) if L is not None]
assert LEVELS, 'no level loaded -- run the sync cell'
LEVELS.sort(key=lambda L: L['eps'])
bulk.add_volume_fractions(cfg, R, LEVELS)
bulk.print_summary(cfg, LEVELS)

## 2 · Figures

In [ ]:
# 1 · Strain diagnostic, all levels
bulk.fig_strain(cfg, R, LEVELS, stem='sweep_strain_diagnostic');

In [ ]:
# 2 · Solvent volume fraction φ_s (mass fraction, Voronoi, λ-calibrated): reference dashed + one plateau curve per level,
#     one figure per axis
bulk.fig_volfrac_sweep(cfg, R, LEVELS, axes=cfg.AXES);

In [ ]:
# 3 · Total stress / P_bath evolution, all levels
bulk.fig_total_stress_sweep(cfg, R, LEVELS);

In [ ]:
# 4 · Solvent partial | polymer partial | total σ_zz evolutions, all levels
bulk.fig_partial_stress_sweep(cfg, R, LEVELS);

In [ ]:
# 5 · Network stress σ'_zz, σ'_xx, σ'_yy: FINAL equilibrated state of every level + reference, along all three axes
bulk.fig_network_stress_sweep(cfg, R, LEVELS, axes=cfg.AXES);

In [ ]:
# 6 · Mean plate pressure histories, linear + log, all levels
bulk.fig_plates_sweep(cfg, R, LEVELS);

In [ ]:
# 7 · Bulk modulus K vs volumetric strain (network per axis and mean, plates).  Lower error bars include δ_sys (RELAX_SYS)
bulk.fig_K_sweep(cfg, R, LEVELS);

In [ ]:
# 7b · Stress vs strain with the ε_vol = 0 readings; dotted line per estimator anchored at its ε_vol = 0 reading, slope from
#      the first SS_FIT_NPTS levels only (the linear regime) -> small-strain K; higher levels' departure = nonlinearity
bulk.fig_stress_strain_sweep(cfg, R, LEVELS);

In [ ]:
# 8 · Isotropy: deviator of the network-stress increment vs volumetric strain, per component and profile axis
bulk.fig_isotropy_sweep(cfg, R, LEVELS);

In [ ]:
# 8b · Network-stress anisotropy over the hold, Δσ'_ii/ΔΠ along cfg.AXES[0], all levels
bulk.fig_ratio_sweep(cfg, R, LEVELS);

In [ ]:
# 8c · Whole-run polymer partial-stress anisotropy (stress_aniso, from step 0; every hold shaded) -- with the free swelling
#      of the slab_with_support run when SWELL_ANISO points at its stress_aniso file
bulk.fig_anisotropy_run(cfg, R, LEVELS, swell_file=SWELL_ANISO);

In [ ]:
# 9 · D_c vs volumetric strain (held-cube fit; hollow dotted = the 1-D fits per axis) + the cube fit of every level
bulk.fig_Dc_sweep(cfg, R, LEVELS);

In [ ]:
# 10 · Closure: K, the triaxial M (M_REF) and G = 3/4 (M − K) vs volumetric strain
bulk.fig_closure_sweep(cfg, R, LEVELS);

In [ ]:
# 11 · (a) bath check per level (feed solid, permeate dashed) vs P_target; (b) solvent expelled, cumulative
bulk.fig_wet_pistons_sweep(cfg, R, LEVELS);

In [ ]:
# 12 · Thermodynamic pressure P_th = −⅓ tr(σ^t) evolution, all levels (dotted = P_bath)
bulk.fig_thermo_pressure_sweep(cfg, R, LEVELS);

In [ ]:
# 13 · Osmotic pressure Π = −⅓ tr(σ′): final equilibrated state of every level + reference, along all three axes
bulk.fig_osmotic_pressure_sweep(cfg, R, LEVELS, axes=cfg.AXES);

## Notes

The geometry, the file layout and the way `K` is measured are described in `bulk_modulus_analysis_single.ipynb`;
they are repeated here so the sweep notebook stands on its own.  Deeper levels need a data file whose vacuum
margins can take the expelled solvent (`add_more_plates_to_gel.ipynb` prints the volumetric strain the margins
absorb) and a longer wall time (one hold per level).

### Geometry and files (2026-09-29)

* **Geometry**: an isolated gel block in its bath; `x`, `y` periodic, `z` closed by two wet NPT-pistons (feed above,
  permeate below) that hold the bath at `P_target`.  Six solvent-transparent plates load the six faces of the gel.
  Atom types: 4 = plate z-lo, 7 = plate z-hi (the *support* and the *load piston* of the two-piston decks),
  8/9 = plates x-lo/x-hi, 10/11 = plates y-lo/y-hi, 5/6 = feed / permeate piston.
* **Core columns**: every profile is taken in a column through the gel — along `z` the cross-section is
  `[plate x-lo + core_margin, plate x-hi − core_margin] × [same in y]`, likewise along `x` and `y` — so each profile
  reads *bath | gel | bath*.  The deck divides the per-atom stress by the **live** column cross-section, so the
  files hold true stresses at every strain (the 2026-08 deck divided a static atom group by a fixed volume).
* **Files**: along `z` the profile files carry the two-piston names (`sigmazz_polymer_…`); along `x`, `y` they are
  tagged `_alongx`, `_alongy`.  `plate_position`, `plate_force[_avg]` hold one column per plate
  (`zlo zhi xlo xhi ylo yhi`; loads are compressive, `N > 0` when the gel pushes the plate outward);
  `strain_vol` holds the plate and network volumetric strains.  Every level is tagged `_<hold>_c<level>`, the
  reference window `_ref`.
* **Profile figures along another axis**: every profile figure takes `axes=`, e.g.
  `bulk.fig_network_stress(cfg, R, L, axes=('x', 'y', 'z'))`; the files are then tagged `_alongx` / `_alongy`.

### How K is measured

* **Terzaghi**: `σ′ = σᵗ − p_pore`, with `p_pore` the total stress in the **bath bins of the same column**, on
  both sides of the gel (≥ `res_gel_gap_bins` bins outside the plates; along `z` also ≥ `res_wall_margin` = 3 σ from
  the wet pistons, whose sheets deplete the solvent and keep half of their pair virial).
* **Osmotic pressure** `Π = −⅓ tr σ′` (positive under compression).  The profiles store compression-positive
  stresses, as every triaxial profile, so in the files it is the mean of the three `σ′_ii`.
* **K_network** `= (⟨Π⟩_int,plateau − Π_ref)/ε_vol`: plateau-averaged profile (last `plateau_frac` of the hold),
  mean over the wall-trimmed interior bins, one estimate per profile axis.  The headline value is the mean of the
  axes; its CI is the mean of their half-widths (the columns overlap at the gel centre, so they are not independent).
* **K_plates** `= (⟨P̄⟩_plateau − P̄_ref)/ε_vol`, `P̄` = mean over the six plates of load / face area, read over the
  longest drift-free trailing window (circular block bootstrap) — the counterpart of `M_piston`.
  `PLATE_AREA='gap'` takes the face enclosed by the other two plate pairs; `'bb'` the live bounding-box face.
* **Increments**: both estimators subtract **their own** `ε_vol = 0` reading (`K_SUBTRACT_REF`), measured in the
  reference window (Phase 1.5), exactly as `M_SUBTRACT_REF`.
* **Strain**: `ε_vol` is the *measured* plate strain `1 − Π(1 − closure_i/L0_i)`; `L0_i` = seated plate gap −
  2 `contact_gap`.  The network's own `V_Rg` and `V_BB` strains are printed next to it.  The plates reach the
  network through its sparse outer shell, so the network strain can fall short of the plate strain;
  `K_net(Rg)` is the same stress increment over the network's own Rg strain; with `K_STRAIN='disp'` (default) the headline K already divides by the network strain from the steady displacement-profile slopes (`disp_*_polymer_cum`, decks since 2026-10-05), and `'rg'` or `'plate'` select the other denominators.
* **Seating** (`SEAT_MODE` in the batch): `rg` seats the plates on the Rg-based faces of the gel, `bb` on the
  outermost beads.  The bounding box of the 12-cell cube is 3–5 σ per side wider than its Rg edge while a 10 %
  level closes each face by ~1.2 σ, so `bb` seating spends the level on the shell.  With `rg` the folded shell
  carries a preload, which is the `P̄_ref` every K subtracts.
* **Unrelaxed hold** (`RELAX_SYS`): an exponential tail fitted to the mean plate pressure; the plateau's excess
  over the fitted asymptote, / ε_vol, is `δ_sys`, added to the **lower** bound of both K.
* **Isotropy**: see *Network anisotropy* below.  Under isotropic compression the deviator of `Δσ′` vanishes by symmetry — there is no `G` in this
  measurement, which is why the anisotropy-ratio and `G` figures of the triaxial notebooks have no counterpart
  here.  A deviator in the isotropy figure is a loading defect (unequal plates, a gel that turned).
* **D_c**: the held-cube consolidation fit, one `D_c` for the three axes — see *D_c: the held-cube fit* below.
* **Closure**: with `M_REF` = the `M` of the same gel from the triaxial notebooks, `G = ¾(M − K)` and
  `κ = D_c/M`; `G_REF` (shear or triaxial notebook) is drawn next to it.

### What replaced the 2026-08 notebook

`archive/bulk_modulus_analysis.ipynb` derived an osmotic `K(t) = −V dΠ/dV` from the whole-run Rg path (its
"active" mask also caught the NPH equilibration), and a drained `K` from a fit over stages recovered by
step-gap clustering with a fixed count of 3, without the reference window the deck recorded.  Here every level is
its own file set, the reference window is the zero of every modulus, and no stage count is assumed anywhere.
### Network anisotropy

* **Over the hold** (`fig_ratio`): `σ′_ii/Π` in the gel interior along every profile axis (1 = isotropic network
  stress) and the increments `Δσ′_ii/ΔΠ` from the ε_vol = 0 reference — the response to the plates alone.
* **Over the whole run** (`fig_anisotropy_run`): `σ_p,xx/σ_p,zz`, `σ_p,yy/σ_p,zz` from the `stress_aniso` file the
  deck writes from step 0 (settle, seating, reference, every level; the holds are shaded).  `slab_with_support`
  writes the same file, so `SWELL_ANISO` = its `stress_aniso_<run>.dat` draws the free swelling of the isolated gel in
  a first panel — for an isolated gel this file and the Rg edges `L_x, L_y, L_z` are what measure the network's
  anisotropy; the box faces are not the gel's faces (see the barostat note in `slab_with_support.lmp`).

### Volume fractions in the core columns

`add_volume_fractions` tessellates `traj_ref` (`REF_VOR_FRAMES` frames) and `traj_stress` of every level
(`VOR_MAX_FRAMES` frames in the plateau + `VOR_EVO_FRAMES` frames over the whole hold) once per frame — the
periodic voro++ pass of `lib/volfrac.py` — and bins the solvent cell volumes inside the core column along `x`, `y`
and `z` (`core_margin` inside the plates, as the deck's stress columns).  Three estimators, as in the triaxial
notebooks: mass fraction `ρ_s/ρ_bath` from the density profiles, Voronoi `φ_s^vor` (column bin volume), and the
λ-calibrated `φ_s^cal = λ(φ_p, P)·φ_s^vor` (exact-saturation norm; `P` per bin from `P_CAL_MODE`: `const` = `P_CAL`,
`pore` = the axis' bath baseline, `thermo` = the local `P_th`).  `fig_volfrac` shows reference vs plateau per axis,
`fig_volfrac_evolution` the evolution over the hold along the three axes.  Frames are cached in
`flow_data_local/bulk/<RUN_ID>/vor_cache/`.

### D_c: the held-cube fit

`fit_Dc_cube` is the three-dimensional form of the triaxial held-slab fit.  Boundary conditions: every face is
drained at the bath pressure (the feed BC of the compression fit: `u″ = −q/D_c` and equal pore pressure on both
faces of every axis, so the strain modes are the periodic `cos 2mπζ` family) and the displacement vanishes at the
centre (the plates close symmetrically about it).  The dilatation relaxes in `cos(2lπζ_x) cos(2mπζ_y) cos(2nπζ_z)`
at `4π² D_c (l²/L_x² + m²/L_y² + n²/L_z²)`; the core-column displacement along an axis therefore contains only the
antisymmetric `sin(2lπζ_a)` shapes (no `cos − 1` family), each carrying the transverse drainage in its decay.  One
`D_c` is fitted to the hold-referenced `u_a/L_a` of the three axes at once (free amplitudes, linear least squares at
each `D_c`, bounded scalar search, exactly the slab fit's machinery).  `DC_N_T = 0` (default) keeps the pure modes
`(l,0,0)`, whose rates equal the slab's `4l²π²D_c/L²` and which are identified one rate per shape; `DC_N_T = 1`
adds the mixed modes `(l,m,n)` — they share the shape `sin 2lπζ` with the pure ones, so with free amplitudes `D_c` is
then pinned only through the multi-`l` structure (a single-rate signal is degenerate between `D_c` and `D_c/2`): use
it as a cross-check.  The 1-D per-axis fits are printed beside the cube fit; `τ₁ = L_max²/(4π² D_c)` is the slowest
mode, `(1,0,0)` along the longest edge, and sizes the hold check as before.